# Error analysis by group — ENEM Redação model (2024 test)

**Why slice-based evaluation matters**
- **Fairness:** Metrics on the full test set can hide worse performance for some regions or demographic groups. Evaluating by UF and by profile (sexo + renda) surfaces where the model may be less reliable or unfair.
- **Deployment risk:** Knowing which slices have high MAE/RMSE helps prioritize monitoring, targeted data collection, or model improvements before rolling out to high-stakes use.

**How this guides decision-making**
- **Where the model is reliable:** Slices with lower MAE can support decisions with more confidence.
- **Where to be cautious:** High-error slices suggest either more uncertainty in predictions or systematic bias; avoid over-relying on point estimates there without additional checks.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import numpy as np

DIR_REPORTS = ROOT / "reports"
DIR_FIGURES = ROOT / "figures"
PATH_PRED = DIR_REPORTS / "predictions_2024.parquet"
DIR_FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
spark = (
    SparkSession.builder.appName("ENEM-Error-Analysis")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
df = spark.read.parquet(str(PATH_PRED))

In [ ]:
# Standardize columns: y_true, y_pred; derive renda_bucket if missing
if "y_true" not in df.columns:
    df = df.withColumn("y_true", F.col("redacao") if "redacao" in df.columns else F.lit(None).cast("double"))
if "y_pred" not in df.columns:
    df = df.withColumn("y_pred", F.col("prediction") if "prediction" in df.columns else F.lit(None).cast("double"))
if "renda_bucket" not in df.columns and "faixa_renda" in df.columns:
    df = df.withColumn(
        "renda_bucket",
        F.when(F.col("faixa_renda").isin("A", "B"), F.lit("low"))
        .when(F.col("faixa_renda").isin("C", "D", "E", "F", "G", "H"), F.lit("mid"))
        .otherwise(F.lit("high")),
    )
elif "renda_bucket" not in df.columns:
    df = df.withColumn("renda_bucket", F.lit("unknown"))
if "sg_uf_residencia" not in df.columns and "uf" in df.columns:
    df = df.withColumnRenamed("uf", "sg_uf_residencia")
if "tp_sexo" not in df.columns and "sexo" in df.columns:
    df = df.withColumnRenamed("sexo", "tp_sexo")

df = df.withColumn("abs_error", F.abs(F.col("y_true") - F.col("y_pred")))
df = df.withColumn("squared_error", F.pow(F.col("y_true") - F.col("y_pred"), 2))
df = df.filter(F.col("y_true").isNotNull() & F.col("y_pred").isNotNull())
df.printSchema()

## Tables: RMSE/MAE by UF and by profile

In [ ]:
# A) By UF
by_uf = (
    df.groupBy("sg_uf_residencia")
    .agg(
        F.sqrt(F.mean("squared_error")).alias("rmse"),
        F.mean("abs_error").alias("mae"),
        F.count("*").alias("n"),
    )
    .filter(F.col("sg_uf_residencia").isNotNull())
)
by_uf.orderBy(F.desc("mae")).show(27, truncate=False)

worst_10_uf = by_uf.orderBy(F.desc("mae")).limit(10)
best_10_uf = by_uf.orderBy(F.asc("mae")).limit(10)
print("Top 10 worst UFs by MAE:")
worst_10_uf.show(10, truncate=False)
print("Top 10 best UFs by MAE:")
best_10_uf.show(10, truncate=False)

In [ ]:
# B) By profile (sexo + renda_bucket)
df = df.withColumn("profile", F.concat_ws("_", F.col("tp_sexo"), F.col("renda_bucket")))
by_profile = (
    df.groupBy("profile", "tp_sexo", "renda_bucket")
    .agg(
        F.sqrt(F.mean("squared_error")).alias("rmse"),
        F.mean("abs_error").alias("mae"),
        F.count("*").alias("n"),
    )
)
by_profile.orderBy(F.desc("mae")).show(30, truncate=False)

worst_10_profile = by_profile.orderBy(F.desc("mae")).limit(10)
best_10_profile = by_profile.orderBy(F.asc("mae")).limit(10)
print("Top 10 worst profiles by MAE:")
worst_10_profile.show(10, truncate=False)
print("Top 10 best profiles by MAE:")
best_10_profile.show(10, truncate=False)

In [ ]:
# C) Optional: UF + renda_bucket (if not too sparse)
by_uf_renda = (
    df.groupBy("sg_uf_residencia", "renda_bucket")
    .agg(
        F.sqrt(F.mean("squared_error")).alias("rmse"),
        F.mean("abs_error").alias("mae"),
        F.count("*").alias("n"),
    )
    .filter(F.col("sg_uf_residencia").isNotNull())
)
by_uf_renda.orderBy(F.desc("mae")).show(20, truncate=False)
print("Sparsity: groups with n < 100:", by_uf_renda.filter(F.col("n") < 100).count())

## Charts: MAE by UF and by profile (top 15 worst)

In [ ]:
# Bar chart: MAE by UF (top 15 worst)
rows_uf = by_uf.orderBy(F.desc("mae")).limit(15).collect()
ufs = [r.sg_uf_residencia or "NA" for r in rows_uf]
maes_uf = [r.mae for r in rows_uf]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ufs))
bars = ax.bar(x, maes_uf, color="#e74c3c", edgecolor="#c0392b", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(ufs, rotation=45, ha="right")
ax.set_ylabel("MAE")
ax.set_xlabel("UF")
ax.set_title("MAE by UF (top 15 worst, 2024 test)")
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle="-", alpha=0.7)
plt.tight_layout()
plt.savefig(DIR_FIGURES / "error_by_uf.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bar chart: MAE by profile (top 15 worst)
rows_profile = by_profile.orderBy(F.desc("mae")).limit(15).collect()
profiles = [r.profile or "NA" for r in rows_profile]
maes_profile = [r.mae for r in rows_profile]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(profiles))
bars = ax.bar(x, maes_profile, color="#3498db", edgecolor="#2980b9", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(profiles, rotation=45, ha="right")
ax.set_ylabel("MAE")
ax.set_xlabel("Profile (sexo_renda_bucket)")
ax.set_title("MAE by profile (top 15 worst, 2024 test)")
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle="-", alpha=0.7)
plt.tight_layout()
plt.savefig(DIR_FIGURES / "error_by_profile.png", dpi=150, bbox_inches="tight")
plt.show()

## Save CSVs to reports/

In [ ]:
DIR_REPORTS.mkdir(parents=True, exist_ok=True)
pdf_uf = by_uf.orderBy("sg_uf_residencia").toPandas()
pdf_profile = by_profile.orderBy("profile").toPandas()
pdf_uf.to_csv(DIR_REPORTS / "error_by_uf.csv", index=False)
pdf_profile.to_csv(DIR_REPORTS / "error_by_profile.csv", index=False)
print("Saved reports/error_by_uf.csv and reports/error_by_profile.csv")